## NLP-Text Classification

Here we build a simple NLP-style text classification pipeline using people's names to predict gender with logistic regression.

In [ ]:
# Import libraries
import string
import nltk
import numpy as np

In [ ]:
nltk.download('names')

[nltk_data] Downloading package names to /root/nltk_data...
[nltk_data]   Unzipping corpora/names.zip.


True

In [ ]:
names = [[name.lower(), 'male'] for name in nltk.corpus.names.words('male.txt')]
names += ([[name.lower(), 'female'] for name in nltk.corpus.names.words('female.txt')])
np.random.shuffle(names) # shuffling makes the dataset more mixed and suitable for learning

## Labeled Corpus
- Input text = name
- Target label = gender

In [ ]:
names[2]

['delcina', 'female']

In [ ]:
# Exercise find the male and female names of maximum length.
maxf, maxm = 0, 0
indf, indm = 0, 0


In [ ]:
#create dataframe of features
import pandas as pd
feature_dlist =[]
pos = []
MAXLEN = 15
for i in range (MAXLEN):
    pos.append('pos'+str(i+1))
feat =['name']+ pos + ['gender']
for name_gender in names:
    cl =list(name_gender[0])+['_']*(MAXLEN-len(name_gender[0]))

    tmp_dict = {}
    if name_gender[1] == 'female':
        featV = [name_gender[0]] + cl + [1]
        for j in range(len(featV)):
            tmp_dict.update({feat[j]:featV[j]})
    else:
        featV = [name_gender[0]] + cl + [0]
        for j in range(len(featV)):
            tmp_dict.update({feat[j]:featV[j]})
    feature_dlist.append(tmp_dict)
df2 = pd.DataFrame(feature_dlist)


In [ ]:
df2.head(10)

,name,pos1,pos2,pos3,pos4,pos5,pos6,pos7,pos8,pos9,pos10,pos11,pos12,pos13,pos14,pos15,gender
0,oren,o,r,e,n,_,_,_,_,_,_,_,_,_,_,_,0
1,jean-pierre,j,e,a,n,-,p,i,e,r,r,e,_,_,_,_,0
2,delcina,d,e,l,c,i,n,a,_,_,_,_,_,_,_,_,1
3,valeria,v,a,l,e,r,i,a,_,_,_,_,_,_,_,_,1
4,dona,d,o,n,a,_,_,_,_,_,_,_,_,_,_,_,1
5,kizzee,k,i,z,z,e,e,_,_,_,_,_,_,_,_,_,1
6,cathe,c,a,t,h,e,_,_,_,_,_,_,_,_,_,_,1
7,mort,m,o,r,t,_,_,_,_,_,_,_,_,_,_,_,0
8,benedikta,b,e,n,e,d,i,k,t,a,_,_,_,_,_,_,1
9,barnabe,b,a,r,n,a,b,e,_,_,_,_,_,_,_,_,0


In [ ]:
df2.iloc[1,2]

'e'

In [ ]:
def create_features(df, feat_cols):
    col = df.columns
    df2 = df[[col[i] for i in feat_cols]]
    return df2
df_feat = create_features(df2, [0, 1, 2, -1])
df_feat

,name,pos1,pos2,gender
0,oren,o,r,0
1,jean-pierre,j,e,0
2,delcina,d,e,1
3,valeria,v,a,1
4,dona,d,o,1
...,...,...,...,...
7939,sander,s,a,0
7940,royal,r,o,0
7941,celinka,c,e,1
7942,johnathon,j,o,0


In [ ]:
def create_features2(name_list, feat_list):
    features =[]
    for i in feat_list:
        if i < 0:
            features.append('pos'+str(i))
        else:
            features.append('pos'+str(i+1))
    features.append('gender')
    tmpdict = {}
    feat = []
    for i in range(len(features)):
        feat.append([])
    for name in name_list:
        for i in range(len(feat_list)):
            feat[i].append(name[0][feat_list[i]])
        if name[1]=='female':
            feat[-1].append(1)
        else:
            feat[-1].append(0)
    tmpdict = {}
    for i in range(len(features)):
        tmpdict.update({features[i]:feat[i]})
    df = pd.DataFrame(tmpdict)
    return df
df3 =  create_features2(names, [-2,-1])
df3.head(10)

,pos-2,pos-1,gender
0,e,n,0
1,r,e,0
2,n,a,1
3,i,a,1
4,n,a,1
5,e,e,1
6,h,e,1
7,r,t,0
8,t,a,1
9,b,e,0


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import scipy.sparse
ct = ColumnTransformer([('oneHot', OneHotEncoder(), [0, 1]) ], remainder='passthrough')
Xt3 = ct.fit_transform(df3).astype(float)
X_temp = Xt3.todense()
X = np.array(X_temp, dtype=np.float32)
type(Xt3), X.dtype

(scipy.sparse._csr.csr_matrix, dtype('float32'))

In [ ]:
X[0,:]

array([0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0.], dtype=float32)

**Exercise 2.**

1. What is one-hot encoding?
2. Write a function to create one-hot encoding from scratch.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
X1 = X[:, 0:52]
y = X[:,-1]
X_train, X_test, y_train, y_test = train_test_split(X1, y, test_size = 0.30, random_state = 100)
logi_regr = LogisticRegression(max_iter=300)
logi_regr.fit(X_train, y_train)
logi_regr.score(X_test, y_test)


1.0

In [ ]:
logi_regr.predict(X_test)

array([1., 1., 1., ..., 1., 0., 0.], dtype=float32)

## Create Sample test

In [48]:
test_names = [['Christian','male'], ['Mark', 'male'], ['Abby', 'Female'], ['Brenda', ['Female']]]
# Using a lambda function and map to convert names and gender labels to lowercase
test_names = list(map(lambda name_gender_pair: [
    name_gender_pair[0].lower(),
    name_gender_pair[1][0].lower() if isinstance(name_gender_pair[1], list) else name_gender_pair[1].lower()
], test_names))

In [49]:
np.random.shuffle(test_names)

In [50]:
test_names

[['mark', 'male'],
 ['christian', 'male'],
 ['abby', 'female'],
 ['brenda', 'female']]

In [51]:
df_test =  create_features2(test_names, [-2,-1])

In [52]:
df_test

,pos-2,pos-1,gender
0,r,k,0
1,a,n,0
2,b,y,1
3,d,a,1


In [27]:
df3.head()

,pos-2,pos-1,gender
0,e,n,0
1,r,e,0
2,n,a,1
3,i,a,1
4,n,a,1


In [58]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import scipy.sparse
# Do NOT re-instantiate ct here. Use the ct that was fitted on df3 in cell WeIXkh_Jl60a.
# ct = ColumnTransformer([('oneHot', OneHotEncoder(), [0, 1]) ], remainder='passthrough')
Xtest = ct.transform(df_test).astype(float) # Use transform, not fit_transform
# Xtest is already a numpy.ndarray, so .todense() is not needed and causes an error.
X_temp = Xtest
X = np.array(X_temp, dtype=np.float32)
type(Xtest), X.dtype

(numpy.ndarray, dtype('float32'))

In [59]:
Xtest[0,:]

array([0., 0., 0., 1., 0., 1., 0., 0., 0.])

In [61]:
# logi_regr.predict(Xtest)

$$\frac{-b\pm \sqrt{b^2 - 4ac}}{2a}$$

In [ ]:
st = "abcd"
st[-1]

'd'